# 07.02 — Peak-Risk Target

Implement and inspect the training-only Peak threshold policy using walk-forward diagnostic labels.

In [1]:
# Import libraries
from pathlib import Path
import sys

In [2]:
# Define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'target_definition.yaml'
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/Codigo/ontario-electricity-peak-risk/configs/target_definition.yaml')

In [3]:
# Import module to manage the Target Definition process
from src.ontario_peak_risk.target_definition.common import (
    load_target_config,
    load_feature_dataset,
    ensure_directories,
)

In [4]:
# Configure the Target Definition process
CONFIG, _ = load_target_config(CONFIG_PATH)
OUTPUT_DIR, REPORTS_DIR, DOCS_DIR = ensure_directories(CONFIG, PROJECT_ROOT)
feature_dataset = load_feature_dataset(CONFIG, PROJECT_ROOT)
feature_dataset.shape

(262944, 100)

In [5]:
# Import module to build the Peak Risk Target matrix
from src.ontario_peak_risk.target_definition.peak_risk_target import (
    build_walk_forward_peak_diagnostics,
)

In [6]:
# Build the Peak Risk Target matrix
PEAK_CFG = CONFIG['target_definition']['peak_risk']
peak_labels, peak_thresholds = build_walk_forward_peak_diagnostics(
    feature_dataset,
    percentile=PEAK_CFG['percentile_threshold'],
    target_column=PEAK_CFG['target_column'],
    grouping_columns=PEAK_CFG['grouping_columns'],
    time_column=PEAK_CFG['diagnostic_time_column'],
    minimum_training_years=PEAK_CFG['minimum_training_years'],
)
peak_labels.shape, peak_thresholds.shape

((210384, 8), (96, 6))

In [7]:
# Display the first few rows of the peak thresholds matrix
peak_thresholds.head(20)

,fsa,season,peak_threshold_kwh,evaluation_year,training_year_start,training_year_end
0,L4T,Fall,14434.4425,2022,2021,2021
1,L4T,Spring,14997.0125,2022,2021,2021
2,L4T,Summer,23769.5000,2022,2021,2021
3,L4T,Winter,15253.3325,2022,2021,2021
4,M5R,Fall,9531.8625,2022,2021,2021
5,M5R,Spring,10031.5800,2022,2021,2021
6,M5R,Summer,12772.5425,2022,2021,2021
7,M5R,Winter,12095.5025,2022,2021,2021
8,M5S,Fall,5241.0025,2022,2021,2021
9,M5S,Spring,5326.1000,2022,2021,2021


In [8]:
# Display the first few rows of the peak labels matrix
peak_labels.head(20)

,fsa,timestamp,year,season,total_consumption_kwh,peak_threshold_kwh,peak_threshold_available,peak_risk_target
0,L4T,2022-01-01 00:00:00,2022,Winter,9544.8,15253.3325,1,0
1,L4T,2022-01-01 01:00:00,2022,Winter,8903.8,15253.3325,1,0
2,L4T,2022-01-01 02:00:00,2022,Winter,8281.0,15253.3325,1,0
3,L4T,2022-01-01 03:00:00,2022,Winter,7882.5,15253.3325,1,0
4,L4T,2022-01-01 04:00:00,2022,Winter,7676.0,15253.3325,1,0
5,L4T,2022-01-01 05:00:00,2022,Winter,7623.2,15253.3325,1,0
6,L4T,2022-01-01 06:00:00,2022,Winter,7778.3,15253.3325,1,0
7,L4T,2022-01-01 07:00:00,2022,Winter,8189.3,15253.3325,1,0
8,L4T,2022-01-01 08:00:00,2022,Winter,8711.7,15253.3325,1,0
9,L4T,2022-01-01 09:00:00,2022,Winter,9666.4,15253.3325,1,0


**Important:** 
- These are leakage-safe diagnostic labels only. Final Peak-Risk labels must be rebuilt inside each modeling training window.